[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Pruebas para dos grupos.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo III: *Inferencia estadística*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se compararán dos grupos con las pruebas *t* de Student y de Welch, la prueba de Mann-Whitney y las pruebas por permutación, según la forma de los datos.

**Al terminar podrás:**

* elegir entre Welch, Mann-Whitney y una permutación según la forma de los datos, con Welch por defecto para comparar medias;
* revisar los supuestos con pruebas y con gráficas;
* reportar la diferencia con su IC del 95%, el p-value y la d de Cohen;
* leer un resultado "no significativo" con el IC, sin concluir que los grupos son iguales.

## 🔁 Antes de empezar: repaso de memoria.

Sin abrir `19_logica_del_contraste.ipynb`, escribe con tus palabras, en tres o cuatro frases, la idea clave de ese *notebook*: qué dicen H0 y H1, qué mide un p-value y qué no mide, y qué quiere decir no rechazar H0. No es un cuestionario ni hay una sola respuesta correcta: se trata de reconstruir la idea de memoria, porque recordar sin mirar ayuda a fijarla. Cuando termines, compáralo con el resumen que está al inicio de la versión con soluciones de este *notebook* y corrige lo que haga falta.

**✍️ Tu respuesta:**

_(escribe aquí)_

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien quiere profundizar. Se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencias.

py311 no cubre inferencia, así que la referencia de la API es la documentación oficial: [*SciPy* · `scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html) (`ttest_ind`, `mannwhitneyu`, `permutation_test`, `chi2_contingency`, `f_oneway`, `levene`), [*statsmodels* · pruebas de proporciones](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportions_ztest.html), [*statsmodels* · `pairwise_tukeyhsd`](https://www.statsmodels.org/stable/generated/statsmodels.stats.multicomp.pairwise_tukeyhsd.html) y [*statsmodels* · `anova_oneway`](https://www.statsmodels.org/stable/generated/statsmodels.stats.oneway.anova_oneway.html).

### Preparación del entorno.

* La siguiente celda importará las bibliotecas y las funciones de *SciPy* y *statsmodels* que se usarán, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.proportion import proportions_ztest

* La siguiente celda leerá `penguins_limpio.csv` (o lo regenerará si no existe), `titanic.csv` y `tips.csv`, y creará `tip_pct`.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio)
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
propinas = pd.read_csv(os.path.join(DATA, "tips.csv")).assign(tip_pct=lambda d: 100 * d["tip"] / d["total_bill"])
pinguinos.shape, titanic.shape, propinas.shape

---
## 1. Dos grupos.

| Prueba | Compara | Supuestos | En *SciPy* |
|---|---|---|---|
| *t* de Student | Medias | Independencia, normalidad aproximada (o *n* grande), **varianzas iguales** | `ttest_ind(a, b)` |
| *t* de Welch | Medias | Independencia, normalidad aproximada; **no** exige varianzas iguales | `ttest_ind(a, b, equal_var=False)` |
| Mann-Whitney | Si un grupo tiende a tener valores mayores (rangos) | Independencia; robusta a atípicos y asimetría, pero con dispersiones o formas muy distintas puede dar un *p* pequeño aunque el centro no cambie | `mannwhitneyu(a, b)` |
| Permutación | Cualquier estadístico (diferencia de medias, de medianas…) | Independencia e intercambiabilidad bajo H0 | `permutation_test((a, b), estadistico)` |

**Decisión práctica:** usar **Welch por defecto** para medias (ojo: en *SciPy*, `ttest_ind` usa por defecto la versión de Student, así que Welch hay que pedirlo con `equal_var=False`); Mann-Whitney o permutación de medianas si hay asimetría fuerte o atípicos con *n* pequeño. La **independencia** no se arregla con ninguna prueba: mediciones repetidas del mismo individuo requieren pruebas pareadas.

**¿Qué hacer si falla un supuesto?**

| Si ocurre esto… | Entonces… |
|---|---|
| Las varianzas de los grupos son distintas | Usar Welch |
| Los datos son muy asimétricos o tienen atípicos y hay pocos datos | Usar Mann-Whitney o una permutación |
| Hay mediciones repetidas del mismo individuo | Usar una prueba para datos **pareados** |
| Los datos no son independientes (por ejemplo, vienen de la misma familia o de la misma mesa) | Ninguna prueba lo arregla: hay que cambiar el diseño |

**Concepto: Student contra Welch.** Las dos calculan lo mismo, la diferencia de medias medida en errores estándar, pero estiman el error estándar de forma distinta:
* **Student** junta las variaciones de los dos grupos en una sola, como si los dos variaran igual. Si el grupo grande varía poco y el pequeño mucho, esa variación combinada queda dominada por el grupo grande y **subestima** el error del grupo pequeño: la prueba se vuelve demasiado confiada.
* **Welch** usa la variación de cada grupo por separado y hace un ajuste para compensarlo. Por eso funciona bien tanto si los grupos varían igual como si no.

**Ejemplo cotidiano:** comparar el tiempo de traslado de 100 personas que van en metro (todas tardan parecido) con el de 10 que van en coche (unas tardan 20 minutos y otras 90). Juntar las dos variaciones haría creer que los del coche son tan predecibles como los del metro.

**Consecuencia: ¿Student o Welch?** Se simularán 5,000 comparaciones con H0 cierta (medias iguales), pero con un grupo pequeño y muy variable (*n* = 10, con desviación estándar de 3) y otro grande y poco variable (*n* = 50, con desviación estándar de 1). Con α = 0.05, la prueba debería equivocarse 5% de las veces.

* La siguiente celda calculará la tasa real de error tipo I de Student y de Welch en ese escenario, con su propio generador de números aleatorios (semilla 401) para que el resultado se repita aunque se vuelva a ejecutar la celda.

In [ ]:
generador_varianzas = np.random.default_rng(401)
pequeno_variable = generador_varianzas.normal(0, 3, size=(5000, 10))
grande_estable = generador_varianzas.normal(0, 1, size=(5000, 50))
{"Student": float((stats.ttest_ind(pequeno_variable, grande_estable, axis=1).pvalue < 0.05).mean()),
 "Welch": float((stats.ttest_ind(pequeno_variable, grande_estable, axis=1, equal_var=False).pvalue < 0.05).mean())}

Student rechaza H0 en torno a 28% de las veces (¡más de cinco veces α!), mientras que Welch se mantiene en ≈ 5%. Cuando las varianzas son iguales, Welch pierde muy poca potencia; por eso es la opción recomendada por defecto (en *SciPy* hay que pedirla con `equal_var=False`).

**Ejemplo:** masa de machos y hembras Adelie. Se revisan los supuestos, se aplican las pruebas y se calcula el tamaño del efecto.

**¿Por qué este orden?** En el flujo del EDA, una comparación de dos grupos se hace en cuatro pasos: (1) describir cada grupo (*n*, centro, dispersión; `13_agregacion`), (2) revisar supuestos, (3) aplicar la prueba y (4) cuantificar la magnitud con su IC. Primero se define la función del tamaño del efecto para usarla al final.

* La siguiente celda definirá `d_cohen`, la diferencia de medias dividida entre la desviación estándar combinada.

In [ ]:
def d_cohen(a, b):
    a, b = np.asarray(a), np.asarray(b)
    varianza_combinada = ((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2)
    return (a.mean() - b.mean()) / np.sqrt(varianza_combinada)

* La siguiente celda separará las masas de machos y hembras Adelie y revisará normalidad (Shapiro-Wilk) e igualdad de varianzas (Levene).

In [ ]:
adelie = pinguinos[pinguinos["species"] == "Adelie"]
machos_adelie = adelie.loc[adelie["sex"] == "Male", "body_mass_g"].to_numpy()
hembras_adelie = adelie.loc[adelie["sex"] == "Female", "body_mass_g"].to_numpy()
{"n": (len(machos_adelie), len(hembras_adelie)),
 "Shapiro machos p": round(float(stats.shapiro(machos_adelie).pvalue), 3),
 "Shapiro hembras p": round(float(stats.shapiro(hembras_adelie).pvalue), 3),
 "Levene p": round(float(stats.levene(machos_adelie, hembras_adelie).pvalue), 3)}

**Interpretación:** con 73 machos y 73 hembras, Shapiro-Wilk no rechaza la normalidad en ningún grupo (*p* = 0.42 y 0.20) y Levene queda en el límite (*p* = 0.051; es la versión con la mediana, que usa *SciPy*; la versión clásica, con el promedio, da 0.035). Como se usará Welch, la igualdad de varianzas no hace falta; la normalidad aproximada y *n* > 30 respaldan la prueba *t*.

> 💡 **Qué preguntan estas dos pruebas.** Shapiro-Wilk pregunta si los datos se parecen a una campana, y Levene pregunta si dos grupos varían parecido; un p-value pequeño dice que «no». Con muy pocos datos casi nunca rechazan, y con muchísimos rechazan por detalles irrelevantes, así que siempre se miran también las gráficas.

**Nota:** las pruebas de normalidad tienen poca potencia con *n* pequeño y rechazan por detalles irrelevantes con *n* grande. Conviene complementarlas con un histograma o un gráfico Q-Q, y recordar que con *n* ≥ 30 por grupo la prueba *t* es robusta a desviaciones moderadas (pero no a datos muy asimétricos, como la tarifa del Titanic en `16_distribucion_muestral.ipynb`).

Un **gráfico Q-Q** coloca tus datos ordenados contra los valores que tendrían si vinieran de una campana. Si los puntos siguen la diagonal, la forma es parecida a la de una campana; si se curvan en los extremos, hay colas pesadas o asimetría.

**Concepto: tres maneras de obtener la distribución bajo H0.**
* **Welch:** con la fórmula de la *t*, que supone normalidad aproximada de las medias (TCL).
* **Mann-Whitney:** ordena todos los valores juntos y compara las **posiciones** (rangos) de cada grupo: si un grupo tiende a tener valores mayores, sus posiciones se acumulan arriba. Si H0 es cierta, las posiciones se reparten al azar entre los dos grupos. No usa los valores, así que es robusta a atípicos (aunque con dispersiones o formas muy distintas puede dar un *p* pequeño sin que cambie el centro). Ojo: compara posiciones (si un grupo tiende a tener valores mayores) y solo equivale a comparar medianas si las dos distribuciones tienen forma parecida; para comparar medianas se usa la permutación de medianas.
* **Permutación:** si H0 es cierta, las etiquetas "macho" y "hembra" son intercambiables. Se revuelven miles de veces, se recalcula la diferencia de medias en cada revuelta y el p-value es la fracción de revueltas con una diferencia al menos tan extrema como la observada. No supone una forma particular de la distribución, pero sí independencia e intercambiabilidad bajo H0; la permutación de promedios sigue siendo sensible a valores extremos y a dispersiones muy distintas entre los grupos.

> 🤔 **Explícalo con tus palabras.** La permutación y Mann-Whitney no suponen una forma de campana. ¿Qué suponen, entonces, y por qué ninguna de las dos arregla datos que no son independientes?

* La siguiente celda aplicará Welch, Mann-Whitney y una prueba por permutación de la diferencia de medias (9,999 permutaciones, `random_state=401`).

In [ ]:
welch = stats.ttest_ind(machos_adelie, hembras_adelie, equal_var=False)
mann_whitney = stats.mannwhitneyu(machos_adelie, hembras_adelie)
permutacion = stats.permutation_test((machos_adelie, hembras_adelie),
                                     lambda a, b, axis=-1: np.mean(a, axis=axis) - np.mean(b, axis=axis),
                                     n_resamples=9999, random_state=401)
pd.DataFrame({
    "estadístico": [welch.statistic, mann_whitney.statistic, permutacion.statistic],
    "p-value": [welch.pvalue, mann_whitney.pvalue, permutacion.pvalue],
}, index=["Welch", "Mann-Whitney", "permutación"])

* La siguiente celda calculará la diferencia de medias con su IC del 95% (Welch) y la d de Cohen.

In [ ]:
ic_welch = welch.confidence_interval(confidence_level=0.95)
(round(float(machos_adelie.mean() - hembras_adelie.mean()), 1), np.round([ic_welch.low, ic_welch.high], 1),
 round(float(d_cohen(machos_adelie, hembras_adelie)), 2))

Las tres pruebas coinciden: *p* minúsculo. El p-value de la permutación (0.0002) es el mínimo que entrega *SciPy* con 9,999 permutaciones en una prueba bilateral (otra convención daría 0.0001): ninguna permutación igualó la diferencia observada, así que el valor real es todavía menor. Lo informativo es la magnitud: los machos pesan unos 675 g más (IC 95%: 573 a 776 g), una d de 2.17, un efecto **muy grande**.

| d de Cohen (orientativo) | 0.2 | 0.5 | 0.8 |
|---|---|---|---|
| Magnitud | Pequeño | Mediano | Grande |

> 🤔 **Explícalo con tus palabras.** Las tres pruebas dan un *p* minúsculo. ¿Qué información agrega el IC de 573 a 776 g que el p-value no da? ¿Y la d de 2.17?

* La siguiente celda imprimirá, con etiquetas en español, las cifras de apoyo de la comparación de machos y hembras Adelie que se acaba de hacer:
  * **Shapiro-Wilk por grupo (W y su *p*):** W va de 0 a 1; cuanto más cerca de 1, más se parecen los datos a una campana.
  * **Levene (estadístico y su *p*):** compara qué tanto se alejan los valores de su centro en cada grupo. *SciPy* usa por defecto la mediana como centro (`center='median'`).
  * **Error estándar de la diferencia de medias:** cuánto cambiaría típicamente esa diferencia si se repitiera el muestreo; es el que usa Welch.
  * **Grados de libertad de Welch:** ajustan la forma de la distribución *t* que usa la prueba; con 73 y 73 datos siempre quedan entre 72 y 144.
  * **Probabilidad de superioridad:** si se toman al azar un macho y una hembra, es la probabilidad de que el macho pese más (los empates cuentan la mitad). Se obtiene al dividir el estadístico U de Mann-Whitney entre el número de parejas posibles (73 × 73).
  * **Asimetría por grupo (`.skew()` de *pandas*):** un valor cercano a 0 indica una forma más o menos simétrica.

In [ ]:
ee_diferencia = np.sqrt(machos_adelie.var(ddof=1) / len(machos_adelie) + hembras_adelie.var(ddof=1) / len(hembras_adelie))
prob_superioridad = mann_whitney.statistic / (len(machos_adelie) * len(hembras_adelie))
grupos_adelie = {"machos": machos_adelie, "hembras": hembras_adelie}
for nombre, grupo in grupos_adelie.items():
    resultado_shapiro = stats.shapiro(grupo)
    print(f"Shapiro-Wilk, {nombre}: W = {resultado_shapiro.statistic:.3f}, p = {resultado_shapiro.pvalue:.3f}")
resultado_levene = stats.levene(machos_adelie, hembras_adelie)
print(f"Levene (center='median', el valor por defecto de SciPy): estadístico = {resultado_levene.statistic:.2f}, "
      f"p = {resultado_levene.pvalue:.3f}")
print(f"Error estándar de la diferencia de medias: {ee_diferencia:.1f} g")
print(f"Grados de libertad de Welch: {welch.df:.1f}")
print(f"Probabilidad de superioridad, U / (n1 · n2): {prob_superioridad:.2f}")
for nombre, grupo in grupos_adelie.items():
    print(f"Asimetría, {nombre} (pandas .skew()): {pd.Series(grupo).skew():.2f}")

**Interpretación:** W queda muy cerca de 1 en los dos grupos (0.983 en machos y 0.977 en hembras) y los dos *p* son grandes (0.42 y 0.20): no hay evidencia de que la forma se aleje de una campana. Levene da 3.87 con *p* = 0.051, en el límite; como Welch no exige varianzas iguales, eso no cambia la decisión. El error estándar de la diferencia es de unos 51 g, así que la diferencia observada (675 g) equivale a unos 13 errores estándar: justo el estadístico *t* de Welch. Los grados de libertad de Welch (135.7) quedan por debajo de 144 porque las dos desviaciones estándar son algo distintas. La probabilidad de superioridad de 0.94 dice que, en unas 94 de cada 100 parejas tomadas al azar, el macho pesa más que la hembra: otra forma, más intuitiva, de leer el tamaño del efecto. Las asimetrías (0.16 y −0.07) están cerca de 0, así que las dos distribuciones son casi simétricas.

📖 **Para profundizar (opcional).** *Las fórmulas de la sección 1.* Ambas pruebas usan $t = (\bar{x}_A - \bar{x}_B) / EE$, con

$$EE_{\text{Student}} = s_p\sqrt{\frac{1}{n_A} + \frac{1}{n_B}} \qquad EE_{\text{Welch}} = \sqrt{\frac{s_A^2}{n_A} + \frac{s_B^2}{n_B}}$$

donde $s_p$ es la desviación estándar combinada (ver `19_logica_del_contraste.ipynb`). Welch, además, ajusta los grados de libertad. El estadístico de Mann-Whitney es $U = R_A - n_A(n_A + 1)/2$, donde $R_A$ es la suma de los rangos del grupo A.

**💡 Ideas clave de la sección 1:**
* Welch por defecto; Mann-Whitney o permutación con asimetría o atípicos.
* Los supuestos se revisan con pruebas **y** gráficas; la independencia no se puede "probar", se razona sobre el diseño.
* Se reporta la diferencia con su IC y la d de Cohen, no solo el p-value.

### Ejercicio 1 · "No significativo" no es "igual".

**🎯 Objetivo:** aplicar Welch con su IC y su tamaño del efecto a un caso con p-value grande. **Qué se espera aprender:** a leer el IC para decir qué diferencias siguen siendo compatibles con los datos cuando no se rechaza H0.

La masa de los Chinstrap y la de los Adelie ya se comparó en `19_logica_del_contraste.ipynb`. Aquí se usará otra medida de las mismas dos especies: la **profundidad del pico** (`bill_depth_mm`, el grosor del pico de arriba abajo, en milímetros). Hay 68 Chinstrap y 151 Adelie con dato, suficientes para que Welch funcione bien. Los datos son observados: el resultado describe cómo se asocian la especie y el pico, no una causa.

Sigue estos pasos con el *DataFrame* `pinguinos`:

1. **Hipótesis.** Escribe en palabras H0 y H1 (prueba bilateral).
2. **Describe los grupos.** Separa `bill_depth_mm` de los Chinstrap y de los Adelie, sin faltantes, y calcula *n*, media y desviación estándar de cada especie.
3. **Prueba de Welch.** Aplica `stats.ttest_ind(..., equal_var=False)` y obtén el IC del 95% de la diferencia de medias (Chinstrap menos Adelie) con `.confidence_interval()`. Con α = 0.05, decide si se **rechaza** o **no se rechaza** H0.
4. **Tamaño del efecto.** Calcula la d de Cohen con la función `d_cohen` de la sección 1.
5. **Conclusión.** ¿Se puede afirmar que las dos especies tienen el pico igual de profundo? Usa el IC para decir qué diferencias siguen siendo compatibles con los datos y compáralas con la desviación estándar de cada especie.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### Ejercicio 2 · Cuenta total en la cena y en la comida.

**🎯 Objetivo:** elegir pruebas robustas para una variable asimétrica con atípicos. **Qué se espera aprender:** a comparar posiciones con Mann-Whitney y **medianas** con una permutación, y a reportar los centros de cada grupo.

El *DataFrame* `propinas` (archivo `tips.csv`) registra 244 mesas de un restaurante: la cuenta, la propina, el día, el horario, el tamaño del grupo y si en la mesa había fumadores. Aquí se usará la **cuenta total** (`total_bill`, en dólares) de las mesas de la cena (`time == "Dinner"`) y de la comida (`time == "Lunch"`). Son 176 cenas y 68 comidas. La cuenta es asimétrica y tiene algunas mesas muy caras (las medidas robustas se vieron en los *notebooks* `04`, `11` y `12`), así que el promedio puede engañar. Los datos son observados: además, casi todas las comidas son de jueves y casi todas las cenas, de viernes a domingo, así que el horario viene junto con el día.

Sigue estos pasos con el *DataFrame* `propinas`:

1. **Hipótesis.** Escribe en palabras H0 y H1 de la permutación de medianas: hablan de las medianas de la población.
2. **Describe los grupos.** Separa `total_bill` de las cenas y de las comidas y calcula *n*, mediana, media, la cuenta máxima y la asimetría (`pd.Series(...).skew()`) de cada grupo. ¿Qué tan lejos queda la media de la mediana?
3. **Mann-Whitney.** Aplica `stats.mannwhitneyu(cena, comida)` y calcula la probabilidad de superioridad (U entre `len(cena) * len(comida)`), como en la sección 1.
4. **Permutación de medianas e IC.** Aplica `stats.permutation_test` a la **diferencia de medianas** (cena menos comida), con `n_resamples=9999` y `random_state=401`. Calcula también el IC del 95% de esa diferencia con `stats.bootstrap((cena, comida), ..., n_resamples=9999, method="percentile", random_state=401)`, como en `18_bootstrap_e_interpretacion_de_intervalos.ipynb`.
5. **Conclusión.** Con α = 0.05, decide en cada prueba si se **rechaza** o **no se rechaza** H0. Reporta las dos medianas, su diferencia en dólares con su IC y los dos p-values, y escribe la conclusión sin lenguaje causal.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### Reto 1 · Verifica la potencia por simulación.

**🎯 Objetivo:** comprobar por simulación un cálculo de potencia. **Qué se espera aprender:** qué significa en la práctica "80% de potencia" y cuánto cae al reducir *n*.

`TTestIndPower` dijo que con d = 0.5 hacen falta unos 64 casos por grupo para 80% de potencia. Compruébalo: simula 5,000 pares de muestras de tamaño 64 (una con media 0 y otra con media 0.5, ambas con desviación estándar 1; usa `generador = np.random.default_rng(401)`) y calcula la fracción de pruebas de Welch con *p* < 0.05. Repite con *n* = 20.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## ✅ Cierre.

**Práctica:** terminar los ejercicios pendientes de este *notebook* y, si es posible, un reto.

**Siguiente *notebook*:** `21_proporciones_y_mas_de_dos_grupos.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>